# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeremOzcn/flyrank-ml-internship-submission/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook moves my lane off the 30k-row
starter CSV and onto the warehouse release, and fixes the weakness I named at the end of
`w02_ml_task_framing.ipynb`: *"I would define the feature cutoff and a later outcome window
before training."* Here the feature window ends **2026-03-31** and the label window is
**2026-04**, so nothing the model sees overlaps what it predicts.

Run order matters. Every claim in section 1 has a query under it in section 3.

## 0. Setup — connect to the warehouse

The release is gated. The token is read from the Colab Secrets panel (`HF_TOKEN`) or, when
running locally, from the machine's Hugging Face credential store — **never typed into a cell**,
because this repository is public.

In [ ]:
%pip -q install duckdb pandas scikit-learn

import os, duckdb, pandas as pd, numpy as np

con = duckdb.connect()

# Token: Colab Secrets -> env var -> local HF credential store. Never printed.
HF_TOKEN = None
try:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])
    print('auth: explicit token')
else:
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")
    print('auth: local credential chain')

BASE  = 'hf://datasets/FlyRank/internship-warehouse'
FEAT_MONTH, LABEL_MONTH = '2026-03', '2026-04'

def daily(month):
    return f"read_parquet('{BASE}/fact_content_daily_performance/month={month}/*.parquet')"

DIM_CLIENTS = f"read_parquet('{BASE}/dim_clients.parquet')"

cols = [r[0] for r in con.sql(f'DESCRIBE SELECT * FROM {daily(FEAT_MONTH)} LIMIT 0').fetchall()]
print(f'{len(cols)} columns in fact_content_daily_performance:')
print(cols)

In [ ]:
# Resolve column names from the live schema instead of hard-coding guesses.
def pick(*candidates):
    for c in candidates:
        if c in cols:
            return c
    raise KeyError(f'none of {candidates} found in {cols}')

DATE    = pick('report_date')
CLIENT  = pick('client_id')
CONTENT = pick('content_id')
IMPR    = pick('gsc_impressions', 'impressions')
CLICKS  = pick('gsc_clicks', 'clicks')
POS     = pick('gsc_avg_position', 'avg_position')
GSC_OK  = pick('gsc_data_available')
GA4_OK  = pick('ga4_data_available')

print({'date': DATE, 'client': CLIENT, 'content': CONTENT, 'impressions': IMPR,
       'clicks': CLICKS, 'position': POS, 'gsc_flag': GSC_OK, 'ga4_flag': GA4_OK})

## 1. Unit of analysis + time window

**The contract in plain words — five answers.**

**1. What one row means.** In the warehouse table, one row is one **content item on one day for
one client** (`report_date × client_id × content_id`). That is not the unit I model. My unit is
**one content item (one page) belonging to one client, summarised as it stood on 2026-03-31** —
the moment an editor would build the review queue. I get there by aggregating March's daily rows
up to one row per `(client_id, content_id)`. Section 3 Q1 checks the daily grain is really what I
just said it is.

**2. Which tables.** `fact_content_daily_performance`, two partitions only:
`month=2026-03` for every feature and `month=2026-04` for the label. `dim_clients` for history
coverage. I do **not** use `fact_content_query_90d`: its fixed 90-day window runs through the
end of the snapshot and would overlap my label month — the data dictionary flags exactly this.

**3. Which time window.** Features: **2026-03-01 → 2026-03-31**. Label: **2026-04-01 → 2026-04-30**.
The windows touch but never overlap. 2026-03 is deliberately mid-panel: the `_sample` table is the
final month (June 2026), which is the natural outcome window of any past→future label, so
developing label logic there means developing inside a test window.

**4. What I would rank (the proxy).** `will_decline` = April impressions fall more than 20% below
March impressions. This is the same >20% rule my Week 2 notebook borrowed from `trend_direction`,
but pointed **forward** instead of backward. It is a proxy for *"this page is worth an editor's
attention"*, not evidence that a page needs rewriting and not evidence that refreshing it would
recover traffic. No refresh-benefit outcome exists anywhere in this data.

**5. One thing I deliberately exclude.** **Every measurement taken inside the April label window.**
April impressions, April clicks, April position — all of it is knowable only *after* the decision
moment, so none of it can be a feature. Section 3 breaks this rule on purpose to show what it does
to the score, then removes it.

## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket.

| Field | Bucket | Why |
|---|---|---|
| March impressions, clicks, position, active days, intra-month momentum | **Feature** | Measured on or before 2026-03-31 |
| April impressions | **Label** | The proxy is computed from it. Never a feature |
| `client_id`, `content_id` | **Context** | Pseudonyms. For grouping and splitting only — a grouped split needs `client_id`, and learning from an ID means memorising clients |
| `report_date` | **Context** | Defines the windows; not a signal |
| `gsc_data_available`, `ga4_data_available` | **Context** | Availability gates, applied as filters. Three-valued: `IS TRUE`, never `= FALSE` |
| Anything measured in April | **Excluded** | Future information at the decision moment |
| `fact_content_query_90d` columns | **Excluded** | Its fixed 90-day window overlaps my label month |
| GA4 engagement columns | **Excluded (this pass)** | Zero-filled before a client's `ga4_data_start`; a blind read would score "not tracked yet" as "no engagement". Section 3 Q3 measures how much of my slice that affects |

In [ ]:
buckets = pd.DataFrame([
    ('impressions_mar',   'feature', 'measured 2026-03-01..03-31'),
    ('ctr_mar',           'feature', 'measured 2026-03-01..03-31'),
    ('avg_position_mar',  'feature', 'measured 2026-03-01..03-31'),
    ('active_days_mar',   'feature', 'measured 2026-03-01..03-31'),
    ('momentum_mar',      'feature', 'measured 2026-03-01..03-31'),
    ('impressions_apr',   'label',   'April window - the thing predicted'),
    ('will_decline',      'label',   'derived from April vs March'),
    (CLIENT,              'context', 'grouped split key; never a feature'),
    (CONTENT,             'context', 'row identity; never a feature'),
    (GSC_OK,              'context', 'availability gate, IS TRUE'),
    (GA4_OK,              'context', 'availability gate, IS TRUE'),
], columns=['field', 'bucket', 'why'])
buckets

## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries, one per claim. A contract line without a query next to it is a guess.

**Q1 — the grain.** Claim: one row really is one `report_date × client_id × content_id`.
The probe returns duplicate keys; **zero rows back means the grain holds.**

In [ ]:
q1 = con.sql(f'''
    SELECT {DATE}, {CLIENT}, {CONTENT}, COUNT(*) AS n
    FROM {daily(FEAT_MONTH)}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
''').df()

print(f'Q1 duplicate grain keys found: {len(q1)}  ->',
      'GRAIN HOLDS' if len(q1) == 0 else 'GRAIN VIOLATED')
q1

**Q2 — my slice's size and date span.** Claim: `month=2026-03` is a full calendar month of daily
rows, and my unit (one page per client) is smaller than the row count by roughly the number of
days each page appears.

In [ ]:
q2 = con.sql(f'''
    SELECT COUNT(*)                                        AS daily_rows,
           COUNT(DISTINCT {CONTENT})                       AS distinct_content,
           COUNT(DISTINCT {CLIENT})                        AS distinct_clients,
           COUNT(DISTINCT ({CLIENT} || '::' || {CONTENT})) AS distinct_units,
           MIN({DATE})                                     AS first_day,
           MAX({DATE})                                     AS last_day,
           COUNT(DISTINCT {DATE})                          AS days_covered
    FROM {daily(FEAT_MONTH)}
''').df()
q2.T

**Q3 — availability.** Claim: the availability flags are **three-valued** (TRUE / FALSE / NULL), so
`= FALSE` silently mishandles the NULLs. I filter with `IS TRUE` and show how many rows survive.

In [ ]:
q3 = con.sql(f'''
    SELECT COUNT(*)                                                  AS all_rows,
           SUM(CASE WHEN {GSC_OK} IS TRUE      THEN 1 ELSE 0 END)    AS gsc_is_true,
           SUM(CASE WHEN {GSC_OK} IS FALSE     THEN 1 ELSE 0 END)    AS gsc_is_false,
           SUM(CASE WHEN {GSC_OK} IS NULL      THEN 1 ELSE 0 END)    AS gsc_is_null,
           SUM(CASE WHEN {GA4_OK} IS TRUE      THEN 1 ELSE 0 END)    AS ga4_is_true,
           SUM(CASE WHEN {GA4_OK} IS FALSE     THEN 1 ELSE 0 END)    AS ga4_is_false,
           SUM(CASE WHEN {GA4_OK} IS NULL      THEN 1 ELSE 0 END)    AS ga4_is_null
    FROM {daily(FEAT_MONTH)}
''').df()
print(q3.T)

surv = int(q3.gsc_is_true[0]); tot = int(q3.all_rows[0])
print(f'\nRows surviving `{GSC_OK} IS TRUE`: {surv:,} of {tot:,}  ({surv/tot:.1%})')
print(f'NULL flags that `= FALSE` would have dropped silently: '
      f'{int(q3.gsc_is_null[0]):,} GSC, {int(q3.ga4_is_null[0]):,} GA4')

### Five features (max), built from the March window only

Each one carries its *available when?* line.

| Feature | Knowable at the 2026-03-31 decision moment because… |
|---|---|
| `impressions_mar` | it sums impressions on 2026-03-01..03-31, all of it observed before the cutoff |
| `ctr_mar` | clicks ÷ impressions over the same closed March window |
| `avg_position_mar` | impression-weighted mean of the daily position, March only |
| `active_days_mar` | counts March days with any impression — a coverage signal, no future dates read |
| `momentum_mar` | last 10 days of March vs first 10 days of March; both halves end on or before 03-31 |

Eligibility mirrors Week 2: at least 100 March impressions (a visibility floor, so a page moving
from 2 impressions to 1 is not called a decline), and `gsc_data_available IS TRUE`.

In [ ]:
feat_sql = f'''
    SELECT {CLIENT} AS client_id,
           {CONTENT} AS content_id,
           SUM({IMPR})                                                    AS impressions_mar,
           SUM({CLICKS})                                                  AS clicks_mar,
           SUM(CASE WHEN {IMPR} > 0 THEN 1 ELSE 0 END)                    AS active_days_mar,
           SUM({POS} * {IMPR}) / NULLIF(SUM(CASE WHEN {POS} > 0 THEN {IMPR} ELSE 0 END), 0)
                                                                          AS avg_position_mar,
           SUM(CASE WHEN {DATE} >= DATE '2026-03-22' THEN {IMPR} ELSE 0 END) AS impr_last10,
           SUM(CASE WHEN {DATE} <= DATE '2026-03-10' THEN {IMPR} ELSE 0 END) AS impr_first10
    FROM {daily(FEAT_MONTH)}
    WHERE {GSC_OK} IS TRUE
    GROUP BY 1, 2
    HAVING SUM({IMPR}) >= 100
'''

label_sql = f'''
    SELECT {CLIENT} AS client_id,
           {CONTENT} AS content_id,
           SUM({IMPR}) AS impressions_apr
    FROM {daily(LABEL_MONTH)}
    WHERE {GSC_OK} IS TRUE
    GROUP BY 1, 2
'''

df = con.sql(f'''
    SELECT f.*, COALESCE(l.impressions_apr, 0) AS impressions_apr,
           (l.client_id IS NULL) AS absent_in_april
    FROM ({feat_sql}) f
    LEFT JOIN ({label_sql}) l USING (client_id, content_id)
''').df()

df['ctr_mar']      = df.clicks_mar / df.impressions_mar
df['momentum_mar'] = (df.impr_last10 + 1) / (df.impr_first10 + 1)
df['will_decline'] = (df.impressions_apr < 0.8 * df.impressions_mar).astype(int)

FEATURES = ['impressions_mar', 'ctr_mar', 'avg_position_mar', 'active_days_mar', 'momentum_mar']
df[FEATURES] = df[FEATURES].replace([np.inf, -np.inf], np.nan)

print(f'eligible pages: {len(df):,}   clients: {df.client_id.nunique()}')
print(f'proxy-positive (will_decline): {df.will_decline.mean():.1%}')
df[FEATURES + ['will_decline']].describe().T

### The trap: one label-derived column, on purpose

I add `impressions_apr` — a column measured **inside the label window** — as a sixth feature.
It is the exact thing `will_decline` is computed from. Scores are Precision@50 (my review
capacity from Week 2) and ROC AUC, on a **client-grouped** split so no client appears in both
sides: a page-level split would let the model see sibling pages from the same site.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def score(feature_cols, label='will_decline', k=50, seed=42):
    X, y, g = df[feature_cols], df[label], df.client_id
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3,
                                    random_state=seed).split(X, y, g))
    m = HistGradientBoostingClassifier(random_state=seed).fit(X.iloc[tr], y.iloc[tr])
    p = m.predict_proba(X.iloc[te])[:, 1]
    top = np.argsort(-p)[:k]
    return roc_auc_score(y.iloc[te], p), y.iloc[te].to_numpy()[top].mean()

base_rate = df.will_decline.mean()
auc_honest, p50_honest = score(FEATURES)
auc_leak,   p50_leak   = score(FEATURES + ['impressions_apr'])

print(f'base rate (random selection):      Precision@50 = {base_rate:.3f}')
print(f'HONEST  (5 March features):        AUC = {auc_honest:.3f}   Precision@50 = {p50_honest:.3f}')
print(f'LEAKED  (+ impressions_apr):       AUC = {auc_leak:.3f}   Precision@50 = {p50_leak:.3f}')
print(f'\nthe leak buys {auc_leak - auc_honest:+.3f} AUC — and it is worth nothing,')
print('because on 2026-03-31 nobody knows April.')

In [ ]:
# Delete the leak. The honest number is the one that leaves this notebook.
df = df.drop(columns=['impressions_apr'])
assert 'impressions_apr' not in df.columns

print('leak removed. columns kept:', list(df.columns))
print(f'\nHONEST RESULT — AUC {auc_honest:.3f}, Precision@50 {p50_honest:.3f} '
      f'vs {base_rate:.3f} base rate')

## 4. Data limits

**The named limitation: a page that disappears between March and April is indistinguishable, in
this data, from a page that stayed live and lost all its impressions.** My `LEFT JOIN` scores an
absent page as `impressions_apr = 0`, which makes it a maximal decline. But the daily fact only
accrues rows from a content item's registration day, and a page can leave the table because it was
deleted, de-indexed, re-mapped to a new URL, or simply stopped being served — none of which is
"this page needs a refresh". The query below measures how much of my label rests on that
assumption. Anything material here is a ceiling on what the proxy can mean.

Three more limits I can state but not fix:

- **Unbalanced panel.** Client histories start on different dates, so a single calendar window
  treats a client with 3 months of history the same as one with 17.
- **GA4 is not usable as-is.** Engagement columns are zero-filled before `ga4_data_start`, and the
  flag is three-valued (Q3). I excluded them rather than score "not tracked yet" as "no engagement".
- **The proxy is not the goal.** Decline measures lost impressions, not editorial need. A model
  fitted to it can at best reproduce its definition.

In [ ]:
n_absent = int(df.absent_in_april.sum())
n_zero   = int(((~df.absent_in_april) & (df.will_decline == 1)).sum())

print(f'eligible March pages:                       {len(df):,}')
print(f'  absent from April entirely:               {n_absent:,}  ({n_absent/len(df):.1%})')
print(f'  present in April and labelled declining:  {n_zero:,}')
print(f'\nshare of all positives that rest on the absent==0 assumption: '
      f'{n_absent/max(df.will_decline.sum(),1):.1%}')

# Same score with absent pages dropped - does the assumption carry the result?
kept = df.absent_in_april == False
if kept.sum() > 100 and df.loc[kept, 'will_decline'].nunique() > 1:
    _df_all = df
    df = df.loc[kept].copy()
    auc_present, p50_present = score(FEATURES)
    df = _df_all
    print(f'\nexcluding absent pages: AUC {auc_present:.3f}, Precision@50 {p50_present:.3f}')
    print(f'(vs {auc_honest:.3f} / {p50_honest:.3f} with them included)')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — IDs are pseudonyms, and the token is
      read from a secrets store, never typed into a cell
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**What this notebook establishes:** a contract whose every claim has a query under it, five
features that are all knowable at the decision moment, and a measured demonstration that one
future column inflates the score. **What it does not establish:** that declining pages benefit
from a refresh. No outcome of any editorial action exists in this data.